In [1]:
import random
import time


__IMPORTANT__ = """
    khi obj.attr_ins              se run type(attr_ins).__get__(self, obj, objtype)
    khi obj.attr_ins  = values    se run type(attr_ins).__set__(self, obj, value)
    khi del obj.attr_ins         se run type(attr_ins).__delete__(self, obj)
    
    
    khi dung qua nhieu @property va .setter, nen dung Normal tranh lap lai code
"""

print(f"{'Normal':-^85}")
# Normal
class Normal(object):
    # def __init__(self, label):
    #     print("Normal.__init__")
    #     self.label = label

    #python 3.6 va moi hon
    #khi new des = Normal() => se run __set_name__ thay the cho __init__
    def __set_name__(self, objtype, name):
        print("Normal.__set_name__")
        self.name = name

        """
            name = 'age'
            name = 'salary'
            name = 'rating'
            
            class Programmer:
               pass
               
            normal = Normal()
            Programmer.age = normal
            normal.__set_name__(Programmer, 'age')
            
            normal = Normal()
            Programmer.salary = normal
            normal.__set_name__(Programmer, 'salary')
            ...
            
        """
    
    # run khi <programmer.age> 
    def __get__(self, obj, objtype=None):

        """
        self = Normal()
        obj = Programmer('Thong', 30, 5_000, 4)
        objtype = Programmer
        """

        print("Normal.__get__(self, obj, objtype):")

        return obj.__dict__[self.name] or 0
    
    # run khi <programmer.age = 38>
    def __set__(self, instance, value):
        print("Normal.__set__(self, obj, value):")
        instance.__dict__[self.name] = value

    # run khi <del programmer.age>
    def __delete__(self, obj):
        print("Normal.__delete__")

print("IMPORTANT scope global file")

class Programmer(object):
    print(
            "IMPORTANT trong class Programmer van run binh thuong  cung nhu la scope global file"
        )

    program = f"{'':*<10}class attribute"

    age = Normal()    #run __set_name__(self, objtype, name)
    salary = Normal() #run __set_name__(self, objtype, name)
    rating = Normal() #run __set_name__(self, objtype, name)
    # khi class attrs = Normal trung ten instance attrs, thi se call class attr, default call instance attrs

    def __init__(self, age, salary, rating):
        self.age = age       #run Normal.__set__(self, obj, value):
        self.salary = salary #run Normal.__set__(self, obj, value):
        self.rating = rating #run Normal.__set__(self, obj, value):
        
        self.program = f"{'program':#^50} instance attribute "

print()

pro = Programmer(28, 5000, 10) #run Normal.__set__(self, obj, value): 3 lan
print()
print(f"{pro.age = }") #run Normal.__get__(self, obj, objtype):

print()
print(f"{pro.salary = }")
print(f"{pro.program = }")

---------------------------------------Normal----------------------------------------
IMPORTANT scope global file
IMPORTANT trong class Programmer van run binh thuong  cung nhu la scope global file
Normal.__set_name__
Normal.__set_name__
Normal.__set_name__

Normal.__set__(self, obj, value):
Normal.__set__(self, obj, value):
Normal.__set__(self, obj, value):

Normal.__get__(self, obj, objtype):
pro.age = 28

Normal.__get__(self, obj, objtype):
pro.salary = 5000
pro.program = '#####################program###################### instance attribute '


In [2]:
print(f"{'LazyProperty':-^85}")

class LazyProperty:
    def __init__(self, function):
        print("LazyProperty.__init__")

        self.function = function
        self.funcname = function.__name__

    def __get__(self, obj, objtype=None) -> object:
        print("LazyProperty.__get__")

        obj.__dict__[self.funcname] = self.function(obj)
        # deepthought.meaning_of_life = meaning_of_life(obj)=42
        return obj.__dict__[self.funcname]

    # def __set__(self, obj, value):
    #     pass

class DeepThought:

    @LazyProperty
    def meaning_of_life(self): # run LazyProperty.__get__
        time.sleep(3)
        return 42

    #meaning_of_life = LazyProperty(meaning_of_life)

    #deepthought.meaning_of_life lan dau se run run LazyProperty.__get__
    #deepthought.meaning_of_life lan sau chi lay value=42 tu __dict__
    #voi dieu kien ko co khai bao __set__
    
deepthought = DeepThought()
print(vars(deepthought))

print(deepthought.meaning_of_life)
print(deepthought.meaning_of_life)
print(deepthought.meaning_of_life)

print(vars(deepthought))

------------------------------------LazyProperty-------------------------------------
LazyProperty.__init__
{}
LazyProperty.__get__
42
42
42
{'meaning_of_life': 42}


In [3]:
print(f"{'EvenNumber':-^85}")

class EvenNumber:
    def __set_name__(self, objtype, name):
        print("EvenNumber.__set_name__")
        print(f"objtype: {objtype}\nname: {name}")
        self.name = name

    def __get__(self, obj, objtype=None):
        print("EvenNumber.__get__")
        return obj.__dict__.get(self.name) or 0

    def __set__(self, obj, value):
        print("EvenNumer.__set__")
        obj.__dict__[self.name] = (value if value % 2 == 0 else 0)

class Values:
    value1 = EvenNumber() #run EvenNumber.__set_name__
    value2 = EvenNumber() #run EvenNumber.__set_name__
    value3 = EvenNumber() #run EvenNumber.__set_name__
    

my_values = Values() 
print()
my_values.value1 = 1
my_values.value2 = 4
print()
print(f"{my_values.value1 = }")
print(f"{my_values.value2 = }")

-------------------------------------EvenNumber--------------------------------------
EvenNumber.__set_name__
objtype: <class '__main__.Values'>
name: value1
EvenNumber.__set_name__
objtype: <class '__main__.Values'>
name: value2
EvenNumber.__set_name__
objtype: <class '__main__.Values'>
name: value3

EvenNumer.__set__
EvenNumer.__set__

EvenNumber.__get__
my_values.value1 = 0
EvenNumber.__get__
my_values.value2 = 4


In [4]:
class Property(object):
    
    COUNT = 0

    def __init__(self, fget=None, fset=None, fdel=None, doc=None):
        type(self).COUNT+=1
        print(F"Property.__init__ {type(self).COUNT}")
        
        self.fget = fget # other_func
        self.fset = fset # other_func
        self.fdel = fdel # other_func
        
        if doc is None and fget is not None:
            doc = fget.__doc__
        self.__doc__ = doc
    
    #property_name la cls_attribute cua type(obj).property_name = Property(...)
    
    # obj.property_name
    def __get__(self, obj, objtype=None):
        print("Property.__get__ START")
        
        if obj is None:
            return self
        
        if self.fget is None:
            raise AttributeError("unreadable attribute")
            
        value = self.fget(obj)
        
        print("Property.__get__ END")
        return value
#         return self.fget(obj)
    
    # obj.property_name = value
    def __set__(self, obj, value):
        print("Property.__set__ START")
        
        if self.fset is None:
            raise AttributeError("can't set attribute")
            
        self.fset(obj, value)
        print("Property.__set__ END")

    def __delete__(self, obj):
        if self.fdel is None:
            raise AttributeError("can't delete attribute")
            
        self.fdel(obj)

    # re_getter fget vi khi @Property da co fget roi
    def getter(self, fget):
        print(f"getter {fget.__name__ = }")
        return type(self)(fget, self.fset, self.fdel, self.__doc__) #new Property()

    def setter(self, fset):
        print(f"setter {fset.__name__ = }")
        return type(self)(self.fget, fset, self.fdel, self.__doc__) #new Property()

    def deleter(self, fdel):
        return type(self)(self.fget, self.fset, fdel, self.__doc__) #new Property()

    
class Human:
    
    def __init__(self, name=None, old=None):
        self._name = name
        self._old = old
    
    @Property # name = Property(name) #1.1
    def name(self):
        print("Human.name(self): getter")
        return self._name
    
    @name.setter # name = name.setter(name) #return new Property #1.2
    def name(self, value):
        print("Human.name(self, value): setter")
        self._name = value
        
    print(f"{'':-<50}")
    
    @Property # old = Property(old) #2.1
    def old(self):
        return self._old

    @old.setter # old = old.setter(old) #return new Property #2.2
    def old(self, value):
        self._old = value

print()
print("end class Human")
print(f"{'':#^50}")
    

thong = Human('thong', 28)
print(f"{thong.name}") # run

print()
thong.name = 'dung'
print()
print(thong.name)


Property.__init__ 1
setter fset.__name__ = 'name'
Property.__init__ 2
--------------------------------------------------
Property.__init__ 3
setter fset.__name__ = 'old'
Property.__init__ 4

end class Human
##################################################
Property.__get__ START
Human.name(self): getter
Property.__get__ END
thong

Property.__set__ START
Human.name(self, value): setter
Property.__set__ END

Property.__get__ START
Human.name(self): getter
Property.__get__ END
dung


In [5]:
import types

class Function(object):

    def __get__(self, obj, objtype=None):
        "Simulate func_descr_get() in Objects/funcobject.c"
        
        if obj is None:
            print("Tao la function")
            return self

        print('Tao la method')
        return types.MethodType(self, obj)
        # types.MethodType(self, instance) return self

    def __call__(self, *args, **kwds):
        pass

    
class Human:
    name = 'dung'
    old = 28
    
    def get_name(self):
        return self.name, self.old
        

print(Human.get_name)
print()
print(Human().get_name())

<function Human.get_name at 0x7f760d4043a0>

('dung', 28)


In [6]:
class StaticMethod(object):
    "Emulate PyStaticMethod_Type() in Objects/funcobject.c"

    def __init__(self, f): # @StaticMethod
        self.f = f

    def __get__(self, obj, objtype=None):
        return self.f

In [7]:
class ClassMethod(object):
    "Emulate PyClassMethod_Type() in Objects/funcobject.c"

    def __init__(self, f): # @ClassMethod
        self.f = f

    def __get__(self, obj, klass=None):
        if klass is None:
            klass = type(obj)
        def newfunc(*args):
            return self.f(klass, *args)
        return newfunc